In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import files
import os
import zipfile

uploaded = files.upload()

zip_path = next(
    (name for name in uploaded if name.lower().endswith(".zip")),
    None
)

if zip_path is None:
    raise ValueError("Please upload the Kaggle dataset ZIP file.")

print("Uploaded:", zip_path)
print("Size:", os.path.getsize(zip_path), "bytes")
print("Valid ZIP:", zipfile.is_zipfile(zip_path))

if not zipfile.is_zipfile(zip_path):
    raise ValueError("This file is not a valid ZIP archive.")

EXTRACT_PATH = "/content/skinsakhi_dataset"
os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(EXTRACT_PATH)

TRAIN_DIR = None
TEST_DIR = None

for root, dirs, _ in os.walk(EXTRACT_PATH):
    if "train_set" in dirs and "test_set" in dirs:
        TRAIN_DIR = os.path.join(root, "train_set")
        TEST_DIR = os.path.join(root, "test_set")
        break

if TRAIN_DIR is None:
    raise FileNotFoundError(
        "Could not find train_set and test_set in the extracted dataset."
    )

print("\nDATASET READY!")
print("Training folder:", TRAIN_DIR)
print("Testing folder:", TEST_DIR)
print("Classes:", os.listdir(TRAIN_DIR))

Saving archive (1).zip to archive (1).zip
Uploaded: archive (1).zip
Size: 18126678 bytes
Valid ZIP: True

DATASET READY!
Training folder: /content/skinsakhi_dataset/skin-disease-datasaet/train_set
Testing folder: /content/skinsakhi_dataset/skin-disease-datasaet/test_set
Classes: ['VI-chickenpox', 'BA-impetigo', 'FU-nail-fungus', 'VI-shingles', 'BA- cellulitis', 'FU-athlete-foot', 'PA-cutaneous-larva-migrans', 'FU-ringworm']


In [3]:
from google.colab import drive
import os

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/SkinSakhi"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Google Drive connected!")
print("Model save folder:", SAVE_DIR)

Mounted at /content/drive
Google Drive connected!
Model save folder: /content/drive/MyDrive/SkinSakhi


In [4]:

import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("GPU:", tf.config.list_physical_devices("GPU"))

class_names = sorted([
    name for name in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, name))
])

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=class_names,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=class_names,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

base_model = keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(
    len(class_names),
    activation="softmax"
)(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint_path = os.path.join(SAVE_DIR, "skinsakhi_best.keras")

checkpoint = keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[checkpoint]
)

model = keras.models.load_model(checkpoint_path)

test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

model.save(os.path.join(SAVE_DIR, "skinsakhi_mobilenetv3.keras"))

with open(os.path.join(SAVE_DIR, "skinsakhi_classes.txt"), "w") as f:
    f.write("\n".join(class_names))

print("SUCCESS! Model and labels saved to Google Drive/SkinSakhi.")


GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Found 924 files belonging to 8 classes.
Using 740 files for training.
Found 924 files belonging to 8 classes.
Using 184 files for validation.
Found 233 files belonging to 8 classes.
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - accuracy: 0.1525 - loss: 2.3975
Epoch 1: val_accuracy improved from None to 0.49457, saving model to /content/drive/MyDrive/SkinSakhi/skinsakhi_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SkinSakhi/skinsakhi_best.keras
24/24 ━━━━━━━━━━━━━━━━━━━━ 35s 792ms/step - accuracy: 0.2284 - loss: 2.1318 - val_accuracy: 0.4946 - val_loss: 1.5439
Epoch 2/5
21/24 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4810 - loss: 1.4837
Epoch 2: val_accuracy improved from 0.49457 to 0.73913, saving model to /content/drive/MyDrive/SkinSakhi/skinsakhi_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/SkinSakhi/skin

In [5]:

import os
from tensorflow import keras

MODEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
LABEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_classes.txt"

print("Model exists:", os.path.exists(MODEL_PATH))
print("Labels exist:", os.path.exists(LABEL_PATH))

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = f.read().splitlines()

print("Model loaded successfully!")
print("Number of classes:", len(class_names))
print("Classes:", class_names)


Model exists: True
Labels exist: True
Model loaded successfully!
Number of classes: 8
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']


In [6]:

import numpy as np
import gradio as gr
from tensorflow import keras

MODEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
LABEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_classes.txt"
THRESHOLD = 0.60

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = f.read().splitlines()


def screen_skin(image):
    if image is None:
        return "Please upload an image.", {}

    image = image.convert("RGB")
    image = image.resize((224, 224))

    arr = np.asarray(image, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)

    predictions = model.predict(arr, verbose=0)[0]
    top_index = int(np.argmax(predictions))
    confidence = float(predictions[top_index])

    if confidence < THRESHOLD:
        return (
            "UNKNOWN IMAGE\n\n"
            "No sufficiently confident prediction was produced.\n"
            "Please try a relevant image or consult a healthcare professional.",
            {}
        )

    scores = {
        class_names[i]: float(predictions[i])
        for i in range(len(class_names))
    }

    result = (
        f"Top prediction: {class_names[top_index]}\n\n"
        f"Model score: {confidence * 100:.1f}%\n\n"
        "Experimental result only — not a diagnosis. "
        "Please consult a qualified healthcare professional."
    )

    return result, scores


css = """
.gradio-container {
    max-width: 760px !important;
    margin: auto !important;
}
#title {
    text-align: center;
    color: #0f766e;
    font-size: 34px;
    font-weight: 700;
}
#subtitle {
    text-align: center;
    color: #64748b;
}
"""

with gr.Blocks(
    title="SkinSakhi",
    theme=gr.themes.Soft(
        primary_hue="teal",
        neutral_hue="slate"
    ),
    css=css
) as demo:

    gr.Markdown("# SkinSakhi", elem_id="title")
    gr.Markdown(
        "AI-assisted skin-image screening prototype",
        elem_id="subtitle"
    )

    image_input = gr.Image(
        type="pil",
        label="Upload a skin image",
        height=300
    )

    check_button = gr.Button(
        "Analyse Image",
        variant="primary"
    )

    result_output = gr.Textbox(
        label="Screening Result",
        lines=5,
        interactive=False
    )

    scores_output = gr.Label(
        label="Model Scores",
        num_top_classes=8
    )

    check_button.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=[result_output, scores_output]
    )

    gr.Markdown(
        "SkinSakhi is an experimental research prototype, not a "
        "medical device. The model has not been clinically validated. "
        "Scores are not medical certainty. Do not use predictions "
        "to diagnose or rule out conditions. Seek professional medical advice."
    )

demo.launch(share=True)


/tmp/ipykernel_1404/2153960531.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a453d1089b33c6ae44.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:

import os
import numpy as np
import gradio as gr
from tensorflow import keras

MODEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
LABEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_classes.txt"
THRESHOLD = 0.60

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "Model not found. Make sure Google Drive is mounted."
    )

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError("Class labels file not found.")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = f.read().splitlines()

print("SkinSakhi model loaded!")
print("Number of classes:", len(class_names))


def screen_skin(image):
    if image is None:
        return "Please upload an image.", {}

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Please upload a larger, clearer image.", {}

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)

    predictions = model.predict(arr, verbose=0)[0]

    if (
        len(predictions) != len(class_names)
        or not np.all(np.isfinite(predictions))
    ):
        return "Unable to produce a valid prediction.", {}

    top_index = int(np.argmax(predictions))
    confidence = float(predictions[top_index])
    predicted_class = class_names[top_index]

    scores = {
        class_names[i]: float(predictions[i])
        for i in range(len(class_names))
    }

    if confidence < THRESHOLD:
        result = (
            "UNCERTAIN RESULT\n\n"
            f"Top model prediction: {predicted_class}\n"
            f"Model score: {confidence * 100:.1f}%\n\n"
            "The score is below 60%, so this prediction is uncertain. "
            "The model may also be wrong. Please consult a qualified "
            "healthcare professional."
        )
    else:
        result = (
            f"Top model prediction: {predicted_class}\n"
            f"Model score: {confidence * 100:.1f}%\n\n"
            "This is an experimental model prediction, not a diagnosis. "
            "A high score does not guarantee correctness. Please consult "
            "a qualified healthcare professional."
        )

    return result, scores


css = """
.gradio-container {
    max-width: 760px !important;
    margin: auto !important;
}
#title {
    text-align: center;
    color: #0f766e;
    font-size: 34px;
    font-weight: 700;
}
#subtitle {
    text-align: center;
    color: #64748b;
    font-size: 15px;
}
footer {
    display: none !important;
}
"""

with gr.Blocks(
    title="SkinSakhi",
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="slate",
        neutral_hue="slate"
    ),
    css=css
) as demo:

    gr.Markdown("# SkinSakhi", elem_id="title")
    gr.Markdown(
        "AI-assisted skin-image screening prototype",
        elem_id="subtitle"
    )

    image_input = gr.Image(
        type="pil",
        label="Upload a skin image",
        height=300
    )

    check_button = gr.Button(
        "Analyse Image",
        variant="primary"
    )

    result_output = gr.Textbox(
        label="Screening Result",
        lines=6,
        interactive=False
    )

    scores_output = gr.Label(
        label="Model Scores",
        num_top_classes=8
    )

    check_button.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=[result_output, scores_output]
    )

    gr.Markdown(
        "MEDICAL DISCLAIMER: SkinSakhi is an experimental research "
        "prototype, not a medical device. It has not been clinically "
        "validated. Model scores are not medical certainty. Do not use "
        "this tool to diagnose or rule out a condition. Seek professional "
        "medical advice."
    )

demo.launch(share=True)


SkinSakhi model loaded!
Number of classes: 8


/tmp/ipykernel_1404/1330217123.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3bd0898c17e3fb18b8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:

import os
import numpy as np
import gradio as gr
from PIL import Image, ImageStat
from tensorflow import keras

MODEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
LABEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_classes.txt"
THRESHOLD = 0.60

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = f.read().splitlines()


def check_image_quality(image):
    if image is None:
        return False, "Please upload an image."

    if image.width < 100 or image.height < 100:
        return False, "Image too small. Upload a clearer image."

    image = image.convert("RGB")
    small = image.resize((64, 64))
    pixels = np.asarray(small, dtype=np.float32)

    if pixels.std() < 8:
        return False, "UNSUPPORTED IMAGE\n\nThe image appears blank or has very little detail."

    return True, ""


def screen_skin(image):
    valid, message = check_image_quality(image)

    if not valid:
        return message, {}

    image = image.convert("RGB").resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if (
        len(predictions) != len(class_names)
        or not np.all(np.isfinite(predictions))
    ):
        return "Unable to process this image. Please try again.", {}

    top_index = int(np.argmax(predictions))
    confidence = float(predictions[top_index])
    predicted_class = class_names[top_index]

    scores = {
        class_names[i]: float(predictions[i])
        for i in range(len(class_names))
    }

    if confidence < THRESHOLD:
        result = (
            "UNCERTAIN RESULT\n\n"
            f"Top model prediction: {predicted_class}\n"
            f"Model score: {confidence * 100:.1f}%\n\n"
            "No sufficiently confident prediction was produced. "
            "This model may not recognize the uploaded image."
        )
    else:
        result = (
            f"Top model prediction: {predicted_class}\n"
            f"Model score: {confidence * 100:.1f}%\n\n"
            "This is an experimental prediction, not a diagnosis. "
            "A high score does not guarantee correctness."
        )

    return result, scores


css = """
.gradio-container {
    max-width: 760px !important;
    margin: auto !important;
}
#title {
    text-align: center;
    color: #0f766e;
    font-size: 34px;
    font-weight: 700;
}
#subtitle {
    text-align: center;
    color: #64748b;
}
footer {
    display: none !important;
}
"""

with gr.Blocks(
    title="SkinSakhi",
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="slate",
        neutral_hue="slate"
    ),
    css=css
) as demo:

    gr.Markdown("# SkinSakhi", elem_id="title")
    gr.Markdown(
        "AI-assisted skin-image screening prototype",
        elem_id="subtitle"
    )

    image_input = gr.Image(
        type="pil",
        label="Upload a skin image",
        height=300
    )

    check_button = gr.Button(
        "Analyse Image",
        variant="primary"
    )

    result_output = gr.Textbox(
        label="Screening Result",
        lines=6,
        interactive=False
    )

    scores_output = gr.Label(
        label="Model Scores",
        num_top_classes=8
    )

    check_button.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=[result_output, scores_output]
    )

    gr.Markdown(
        "MEDICAL DISCLAIMER: SkinSakhi is an experimental research "
        "prototype, not a medical device. It has not been clinically "
        "validated. Model scores are not medical certainty. Do not use "
        "this tool to diagnose or rule out a condition. Seek professional "
        "medical advice."
    )

demo.launch(share=True)


/tmp/ipykernel_1404/3742362789.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7ae1f7c5cc88875e2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
from google.colab import files
from PIL import Image
import numpy as np

uploaded = files.upload()
image_path = next(iter(uploaded))

image = Image.open(image_path).convert("RGB").resize((224, 224))
arr = np.asarray(image, dtype=np.float32)[None, ...]

predictions = model.predict(arr, verbose=0)[0]
top_indices = np.argsort(predictions)[::-1][:3]

for i in top_indices:
    print(f"{class_names[i]}: {predictions[i] * 100:.2f}%")

KeyboardInterrupt: 

In [10]:
import os
import numpy as np
from tensorflow import keras

folder = "/content/drive/MyDrive/SkinSakhi"

print("Model files:")
for name in os.listdir(folder):
    print(name)

model = keras.models.load_model(
    folder + "/skinsakhi_mobilenetv3.keras"
)

with open(folder + "/skinsakhi_classes.txt") as f:
    class_names = f.read().splitlines()

print("\nNumber of classes:", len(class_names))
print("Classes:", class_names)
print("Model loaded successfully!")

Model files:
skinsakhi_best.keras
skinsakhi_mobilenetv3.keras
skinsakhi_classes.txt

Number of classes: 8
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']
Model loaded successfully!


In [11]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

test_dir = "/content/skinsakhi_dataset/skin-disease-dataset/test_set"

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

predictions = model.predict(test_ds, verbose=1)
y_pred = np.argmax(predictions, axis=1)

print("\nCLASS-WISE PERFORMANCE:\n")
print(classification_report(
    y_true,
    y_pred,
    labels=range(len(test_ds.class_names)),
    target_names=test_ds.class_names,
    zero_division=0
))

print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_true, y_pred))

FileNotFoundError: [Errno 2] No such file or directory: '/content/skinsakhi_dataset/skin-disease-dataset/test_set'

In [12]:
import os

for root, dirs, files in os.walk("/content/skinsakhi_dataset"):
    level = root.replace("/content/skinsakhi_dataset", "").count(os.sep)
    if level <= 3:
        print(root)
        for d in dirs:
            print("  Folder:", d)

/content/skinsakhi_dataset
  Folder: skin-disease-datasaet
/content/skinsakhi_dataset/skin-disease-datasaet
  Folder: test_set
  Folder: train_set
/content/skinsakhi_dataset/skin-disease-datasaet/test_set
  Folder: VI-chickenpox
  Folder: BA-impetigo
  Folder: FU-nail-fungus
  Folder: VI-shingles
  Folder: BA- cellulitis
  Folder: FU-athlete-foot
  Folder: PA-cutaneous-larva-migrans
  Folder: FU-ringworm
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/VI-chickenpox
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/BA-impetigo
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/FU-nail-fungus
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/VI-shingles
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/BA- cellulitis
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/FU-athlete-foot
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/PA-cutaneous-larva-migrans
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/FU-ringworm
/cont

In [13]:
import os
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras

base = "/content/skinsakhi_dataset"

test_dir = None

for root, dirs, files in os.walk(base):
    if os.path.basename(root) == "test_set":
        test_dir = root
        break

print("Test folder:", test_dir)

if test_dir is None:
    print("Test folder not found. Check the dataset extraction.")
else:
    test_ds = keras.utils.image_dataset_from_directory(
        test_dir,
        image_size=(224, 224),
        batch_size=32,
        shuffle=False
    )

    y_true = np.concatenate([y.numpy() for _, y in test_ds])
    probs = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probs, axis=1)

    print("\nCLASS-WISE RESULTS:\n")
    print(classification_report(
        y_true,
        y_pred,
        target_names=test_ds.class_names,
        zero_division=0
    ))

    print("\nCONFUSION MATRIX:\n")
    print(confusion_matrix(y_true, y_pred))



Test folder: /content/skinsakhi_dataset/skin-disease-datasaet/test_set
Found 233 files belonging to 8 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 683ms/step

CLASS-WISE RESULTS:

                            precision    recall  f1-score   support

            BA- cellulitis       0.82      0.94      0.87        33
               BA-impetigo       0.95      0.90      0.92        20
           FU-athlete-foot       0.91      0.97      0.94        32
            FU-nail-fungus       1.00      0.97      0.98        33
               FU-ringworm       0.94      0.65      0.77        23
PA-cutaneous-larva-migrans       0.65      0.60      0.62        25
             VI-chickenpox       1.00      0.94      0.97        34
               VI-shingles       0.79      0.94      0.86        33

                  accuracy                           0.88       233
                 macro avg       0.88      0.86      0.87       233
              weighted avg       0.88      0.88      0.88       233


CONFUSI

In [14]:
import os

train_dir = "/content/skinsakhi_dataset/skin-disease-dataset/train_set"

for name in sorted(os.listdir(train_dir)):
    path = os.path.join(train_dir, name)
    if os.path.isdir(path):
        count = sum(
            1 for f in os.listdir(path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        )
        print(f"{name}: {count} images")


FileNotFoundError: [Errno 2] No such file or directory: '/content/skinsakhi_dataset/skin-disease-dataset/train_set'

In [15]:
import os

base = "/content/skinsakhi_dataset"

for root, dirs, files in os.walk(base):
    if os.path.basename(root) == "train_set":
        train_dir = root
        break
else:
    raise FileNotFoundError("Training folder not found")

print("Correct path:", train_dir)
print("\nTraining images per class:")

for name in sorted(os.listdir(train_dir)):
    path = os.path.join(train_dir, name)
    if os.path.isdir(path):
        count = sum(
            f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
            for f in os.listdir(path)
        )
        print(f"{name}: {count}")

Correct path: /content/skinsakhi_dataset/skin-disease-datasaet/train_set

Training images per class:
BA- cellulitis: 136
BA-impetigo: 80
FU-athlete-foot: 124
FU-nail-fungus: 129
FU-ringworm: 90
PA-cutaneous-larva-migrans: 100
VI-chickenpox: 136
VI-shingles: 130


In [16]:
best_model = keras.models.load_model(
    "/content/drive/MyDrive/SkinSakhi/skinsakhi_best.keras"
)

best_loss, best_acc = best_model.evaluate(test_ds, verbose=0)
print(f"Best model test accuracy: {best_acc * 100:.2f}%")

Best model test accuracy: 87.98%


In [17]:
print("Current model:", model.name)
print("Best model:", best_model.name)
print("Training dataset:", train_dir)
print("Classes:", class_names)

Current model: functional
Best model: functional
Training dataset: /content/skinsakhi_dataset/skin-disease-datasaet/train_set
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']


In [18]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=True,
    seed=42
)

test_dir = "/content/skinsakhi_dataset/skin-disease-dataset/test_set"

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

# Augmentation for training only
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1)
])

# Keep original model safe; fine-tune a copy
improved = keras.models.load_model(
    "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
)

# Freeze feature layers and train the classification head
for layer in improved.layers:
    layer.trainable = False

improved.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train with augmentation
aug_train_ds = train_ds.map(
    lambda x, y: (augment(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

improved.fit(aug_train_ds, validation_data=test_ds, epochs=3)

save_path = "/content/drive/MyDrive/SkinSakhi/skinsakhi_experimental.keras"
improved.save(save_path)

print("Saved separate experimental model:", save_path)

Found 924 files belonging to 8 classes.


FileNotFoundError: [Errno 2] No such file or directory: '/content/skinsakhi_dataset/skin-disease-dataset/test_set'

In [19]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

base = "/content/skinsakhi_dataset"
save_path = "/content/drive/MyDrive/SkinSakhi/skinsakhi_experimental.keras"

train_dir = None
test_dir = None

for root, dirs, files in os.walk(base):
    if os.path.basename(root) == "train_set":
        train_dir = root
    elif os.path.basename(root) == "test_set":
        test_dir = root

if train_dir is None or test_dir is None:
    raise FileNotFoundError(
        "Dataset folders not found. Check whether the dataset is extracted."
    )

print("Training folder:", train_dir)
print("Testing folder:", test_dir)

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=True,
    seed=42
)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("Classes:", train_ds.class_names)

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1)
])

model_path = "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
improved = keras.models.load_model(model_path)

# Keep all original model layers frozen for this experiment
for layer in improved.layers:
    layer.trainable = False

improved.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

aug_train_ds = train_ds.map(
    lambda x, y: (augment(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

history = improved.fit(
    aug_train_ds,
    validation_data=test_ds,
    epochs=3
)

test_loss, test_acc = improved.evaluate(test_ds, verbose=0)
print(f"\nExperimental model test accuracy: {test_acc * 100:.2f}%")

improved.save(save_path)
print("Experimental model saved:", save_path)
print("Original model remains unchanged.")

Training folder: /content/skinsakhi_dataset/skin-disease-datasaet/train_set
Testing folder: /content/skinsakhi_dataset/skin-disease-datasaet/test_set
Found 924 files belonging to 8 classes.
Found 233 files belonging to 8 classes.
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']
Epoch 1/3


/usr/local/lib/python3.13/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


58/58 ━━━━━━━━━━━━━━━━━━━━ 33s 402ms/step - accuracy: 0.7348 - loss: 0.8476 - val_accuracy: 0.8798 - val_loss: 0.6263
Epoch 2/3
58/58 ━━━━━━━━━━━━━━━━━━━━ 8s 138ms/step - accuracy: 0.7695 - loss: 0.8214 - val_accuracy: 0.8798 - val_loss: 0.6263
Epoch 3/3
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 139ms/step - accuracy: 0.7489 - loss: 0.8319 - val_accuracy: 0.8798 - val_loss: 0.6263

Experimental model test accuracy: 87.98%
Experimental model saved: /content/drive/MyDrive/SkinSakhi/skinsakhi_experimental.keras
Original model remains unchanged.


In [20]:
import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

# Load saved model and class names
BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.60
MIN_MARGIN = 0.10

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Class labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Number of class labels does not match model outputs.")

def screen_skin(image):
    if image is None:
        return "Please upload an image.", {}

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Please upload a larger, clearer image.", {}

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try another image.", {}

    top_indices = np.argsort(predictions)[::-1][:3]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )
    margin = confidence - second_score

    scores = {
        class_names[i]: float(predictions[i])
        for i in range(len(class_names))
    }

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        result = (
            "⚠️ UNCERTAIN RESULT\n\n"
            f"Top model suggestion: {class_names[top_index]}\n"
            f"Model score: {confidence * 100:.1f}%\n\n"
            "The model cannot provide a sufficiently clear result. "
            "Please consult a qualified healthcare professional."
        )
    else:
        result = (
            "⚠️ EXPERIMENTAL SCREENING RESULT\n\n"
            f"Model suggestion: {class_names[top_index]}\n"
            f"Model score: {confidence * 100:.1f}%\n\n"
            "This score is not a measure of clinical certainty. "
            "The model may be wrong. This tool cannot diagnose "
            "skin conditions. Please seek professional medical advice."
        )

    return result, scores


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown(
        """
        # 🩺 SkinSakhi
        ### AI-Assisted Skin Condition Screening

        Upload a clear skin image to see an experimental model suggestion.
        """
    )

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload skin image"
            )
            analyze_btn = gr.Button(
                "Analyze Image",
                variant="primary"
            )
            clear_btn = gr.ClearButton(
                [image_input]
            )

        with gr.Column():
            result_output = gr.Textbox(
                label="Screening Result",
                lines=9
            )
            scores_output = gr.Label(
                label="Model Scores",
                num_top_classes=3
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=[result_output, scores_output]
    )

    gr.Markdown(
        """
        ---
        **Medical disclaimer:** SkinSakhi is an experimental educational
        prototype, not a medical device. It cannot confirm or rule out
        any disease. Model scores may be misleading, and unrelated images
        may still receive predictions. Consult a qualified healthcare
        professional for medical assessment.
        """
    )

demo.launch(share=True)

/tmp/ipykernel_1404/643397044.py:82: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aed41535a2a5618d19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

# Model files
BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

# Provisional rejection thresholds
THRESHOLD = 0.70
MIN_MARGIN = 0.15

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError("Saved model not found. Check Google Drive.")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError("Class labels not found. Check Google Drive.")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match the model outputs.")

def screen_skin(image):
    if image is None:
        return "Please upload an image.", {}

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Unable to assess: image is too small.", {}

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image.", {}

    top_indices = np.argsort(predictions)[::-1][:3]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )
    margin = confidence - second_score

    scores = {
        class_names[i]: float(predictions[i])
        for i in range(len(class_names))
    }

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        result = (
            "⚠️ UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain about this image. It may be "
            "unrelated, unclear, or outside the conditions covered "
            "by this experimental model.\n\n"
            "Please do not use this result to make medical decisions. "
            "Consult a qualified healthcare professional."
        )
    else:
        result = (
            "⚠️ EXPERIMENTAL MODEL SUGGESTION\n\n"
            f"Suggested class: {class_names[top_index]}\n"
            f"Model score: {confidence * 100:.1f}%\n\n"
            "This score is not a measure of clinical certainty. "
            "The prediction may be wrong. This tool cannot diagnose "
            "skin conditions. Please consult a qualified healthcare "
            "professional."
        )

    return result, scores


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown("""
    # 🩺 SkinSakhi
    ### AI-Assisted Skin Condition Screening

    Upload a clear image for an experimental model suggestion.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Image"
            )
            analyze_btn = gr.Button(
                "Analyze Image",
                variant="primary"
            )
            clear_btn = gr.ClearButton(
                [image_input]
            )

        with gr.Column():
            result_output = gr.Textbox(
                label="Screening Result",
                lines=9
            )
            scores_output = gr.Label(
                label="Model Scores",
                num_top_classes=3
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=[result_output, scores_output]
    )

    gr.Markdown("""
    ---
    **Medical disclaimer:** SkinSakhi is an experimental educational
    prototype, not a medical device. It cannot confirm or rule out
    disease. Unrelated images may still receive predictions, and
    high scores can be wrong. Seek professional medical advice.
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/3506003416.py:85: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cc0cb18154c930c66.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [22]:

import os
import numpy as np
import gradio as gr
from PIL import Image, ImageStat
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.70
MIN_MARGIN = 0.15

if not os.path.exists(MODEL_PATH) or not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(
        "Model or class labels missing. Check your Google Drive."
    )

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Model output count does not match class labels.")


def basic_image_check(image):
    """
    Reject some obvious document-like images.
    This is only a heuristic, NOT a reliable skin-image detector.
    """
    small = image.convert("RGB").resize((160, 160))
    arr = np.asarray(small)

    # Very low colour variation can indicate blank pages or diagrams.
    channel_std = np.mean(np.std(arr, axis=(0, 1)))

    # Count near-white pixels, common in some document screenshots.
    near_white = np.mean(np.all(arr > 235, axis=2))

    # Count very dark pixels, often found in screenshots/documents.
    dark_pixels = np.mean(np.all(arr < 35, axis=2))

    # These rules only catch some obvious cases and may reject
    # genuine skin photos. They cannot establish that an image is skin.
    if channel_std < 12:
        return False

    if near_white > 0.82 and channel_std < 35:
        return False

    if dark_pixels > 0.85:
        return False

    return True


def screen_skin(image):
    if image is None:
        return "Please upload an image.", {}

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Unable to assess: image is too small.", {}

    if not basic_image_check(image):
        return (
            "IMAGE REJECTED\n\n"
            "This image may be a document, screenshot, or unsuitable image. "
            "Upload a clear, well-lit skin photo instead. "
            "This basic check can make mistakes."
        ), {}

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image.", {}

    top_indices = np.argsort(predictions)[::-1][:3]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )
    margin = confidence - second_score

    scores = {
        class_names[i]: float(predictions[i])
        for i in range(len(class_names))
    }

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        result = (
            "UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain. Try a clear skin photo, or "
            "consult a qualified healthcare professional. "
            "This tool cannot diagnose a condition."
        )
    else:
        result = (
            "EXPERIMENTAL MODEL SUGGESTION\n\n"
            f"Suggested class: {class_names[top_index]}\n"
            f"Model score: {confidence * 100:.1f}%\n\n"
            "The score is not clinical certainty. The prediction may "
            "be wrong. Consult a qualified healthcare professional."
        )

    return result, scores


with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="teal", secondary_hue="blue"),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown("""
    # 🩺 SkinSakhi
    ### AI-Assisted Skin Condition Screening

    Upload a clear skin image for an experimental model suggestion.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload Image")
            analyze_btn = gr.Button("Analyze Image", variant="primary")
            gr.ClearButton([image_input])

        with gr.Column():
            result_output = gr.Textbox(label="Screening Result", lines=8)
            scores_output = gr.Label(
                label="Model Scores",
                num_top_classes=3
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=[result_output, scores_output]
    )

    gr.Markdown("""
    ---
    **Disclaimer:** This is an experimental educational prototype, not
    a medical device. It may misclassify skin photos or accept unrelated
    images. It cannot confirm or rule out disease. Seek professional
    medical advice.
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/4174344063.py:118: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d34752886ef1dc5e5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.70
MIN_MARGIN = 0.15

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


def basic_image_check(image):
    small = np.asarray(
        image.convert("RGB").resize((160, 160)),
        dtype=np.float32
    )

    colour_variation = float(np.mean(np.std(small, axis=(0, 1))))
    near_white = float(np.mean(np.all(small > 235, axis=2)))
    dark_pixels = float(np.mean(np.all(small < 35, axis=2)))

    # Heuristic only; this is not a trained skin-image detector.
    if colour_variation < 12:
        return False

    if near_white > 0.82 and colour_variation < 35:
        return False

    if dark_pixels > 0.85:
        return False

    return True


def screen_skin(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer image."

    if not basic_image_check(image):
        return (
            "IMAGE NOT ACCEPTED\n\n"
            "This image may be unsuitable or document-like. "
            "Please upload a clear skin photo.\n\n"
            "This basic check is not perfect and may make mistakes."
        )

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try another."

    if len(predictions) != len(class_names):
        return "Unable to assess this image."

    top_indices = np.argsort(predictions)[::-1]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )
    margin = confidence - second_score

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        return (
            "UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain about this image. "
            "Try a clearer skin photo or consult a healthcare professional."
        )

    return (
        "EXPERIMENTAL MODEL SUGGESTION\n\n"
        f"Suggested class: {class_names[top_index]}\n\n"
        "This is not a diagnosis. The model can still be wrong, "
        "even when its score is high. Please consult a qualified "
        "healthcare professional for medical advice."
    )


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown("""
    # 🩺 SkinSakhi
    ### AI-Assisted Skin Condition Screening

    Upload a clear skin photo to receive an experimental model suggestion.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Skin Image"
            )

            analyze_btn = gr.Button(
                "Analyze Image",
                variant="primary"
            )

            clear_btn = gr.ClearButton([image_input])

        with gr.Column():
            result_output = gr.Textbox(
                label="Screening Result",
                lines=9,
                interactive=False
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result_output
    )

    gr.Markdown("""
    ---
    **Medical disclaimer:** SkinSakhi is an experimental educational
    prototype, not a medical device. It cannot diagnose or rule out
    disease. Unrelated images may still pass the basic image check.
    Seek professional medical advice.
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/4009253520.py:106: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3459ebd75a1c50b1a1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [24]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

# ---------------- MODEL SETUP ----------------

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.70
MIN_MARGIN = 0.15

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


# ---------------- IMAGE CHECK ----------------

def basic_image_check(image):
    small = np.asarray(
        image.convert("RGB").resize((160, 160)),
        dtype=np.float32
    )

    colour_variation = float(np.mean(np.std(small, axis=(0, 1))))
    near_white = float(np.mean(np.all(small > 235, axis=2)))
    dark_pixels = float(np.mean(np.all(small < 35, axis=2)))

    # Basic visual heuristics only, not a trained image detector.
    if colour_variation < 12:
        return False

    if near_white > 0.82 and colour_variation < 35:
        return False

    if dark_pixels > 0.85:
        return False

    return True


# ---------------- PREDICTION ----------------

def screen_skin(image):
    if image is None:
        return (
            "NO_IMAGE",
            "Please upload a clear skin photo to begin."
        )

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return (
            "NO_IMAGE",
            "The image is too small. Please upload a clearer image."
        )

    if not basic_image_check(image):
        return (
            "REJECTED",
            "This image may be a document or an unsuitable image. "
            "Please upload a clear skin photo."
        )

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return (
            "UNCERTAIN",
            "Unable to assess this image. Please try another image."
        )

    if len(predictions) != len(class_names):
        return (
            "UNCERTAIN",
            "Unable to assess this image."
        )

    top_indices = np.argsort(predictions)[::-1]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )

    margin = confidence - second_score

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        return (
            "UNCERTAIN",
            "The model could not produce a sufficiently clear result. "
            "Try a clearer photo or consult a qualified healthcare professional."
        )

    return (
        "RESULT",
        f"Model suggestion: {class_names[top_index]}"
    )


def update_result(image):
    status, message = screen_skin(image)

    if status == "RESULT":
        return (
            gr.update(value="SCREENING SUGGESTION", visible=True),
            gr.update(value=message, visible=True),
            gr.update(visible=True)
        )

    if status == "UNCERTAIN":
        return (
            gr.update(value="RESULT UNCERTAIN", visible=True),
            gr.update(value=message, visible=True),
            gr.update(visible=True)
        )

    if status == "REJECTED":
        return (
            gr.update(value="IMAGE NOT ACCEPTED", visible=True),
            gr.update(value=message, visible=True),
            gr.update(visible=True)
        )

    return (
        gr.update(value="READY TO BEGIN", visible=True),
        gr.update(value=message, visible=True),
        gr.update(visible=True)
    )


# ---------------- PREMIUM WEBSITE DESIGN ----------------

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap');

.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
    font-family: 'DM Sans', sans-serif !important;
    background: #f4f8f7 !important;
}

body {
    background: #f4f8f7 !important;
}

#hero {
    background: linear-gradient(125deg, #073b3a 0%, #087e78 65%, #16a394 100%);
    border-radius: 26px;
    padding: 38px 38px 34px;
    color: white;
    margin-bottom: 22px;
    box-shadow: 0 14px 35px rgba(5, 75, 70, 0.16);
}

#brand {
    font-family: 'Manrope', sans-serif;
    font-size: 34px;
    font-weight: 800;
    letter-spacing: -1.5px;
    color: #ffffff;
    margin: 0;
}

#hero-subtitle {
    font-size: 16px;
    line-height: 1.7;
    color: #e0f7f3;
    max-width: 640px;
    margin-top: 12px;
}

#hero-tag {
    display: inline-block;
    background: rgba(255,255,255,0.13);
    border: 1px solid rgba(255,255,255,0.25);
    border-radius: 30px;
    padding: 7px 13px;
    color: #eafffb;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 1px;
    margin-bottom: 18px;
}

#main-card {
    background: white;
    border: 1px solid #e3ece9;
    border-radius: 22px;
    padding: 23px;
    box-shadow: 0 8px 28px rgba(18, 56, 49, 0.045);
}

#section-title {
    font-family: 'Manrope', sans-serif;
    color: #123d39;
    font-size: 21px;
    font-weight: 800;
    margin-bottom: 5px;
}

#section-help {
    color: #71817e;
    font-size: 13px;
    line-height: 1.6;
    margin-bottom: 16px;
}

#upload-image {
    border: 2px dashed #b6d9d1 !important;
    border-radius: 17px !important;
    background: #f6fbf9 !important;
    overflow: hidden;
}

#analyze-btn {
    background: linear-gradient(100deg, #087e78, #0c9b8d) !important;
    color: white !important;
    border: none !important;
    border-radius: 13px !important;
    min-height: 52px !important;
    font-weight: 700 !important;
    font-size: 15px !important;
    box-shadow: 0 6px 15px rgba(8, 126, 120, 0.18);
    transition: transform .2s ease, box-shadow .2s ease;
}

#analyze-btn:hover {
    transform: translateY(-1px);
    box-shadow: 0 9px 20px rgba(8, 126, 120, 0.24);
}

#clear-btn {
    border-radius: 12px !important;
    border: 1px solid #dce9e5 !important;
    background: white !important;
}

#result-card {
    background: #ffffff;
    border: 1px solid #e3ece9;
    border-radius: 22px;
    padding: 25px;
    min-height: 290px;
    box-shadow: 0 8px 28px rgba(18, 56, 49, 0.045);
}

#result-label {
    color: #087e78;
    font-size: 11px;
    font-weight: 800;
    letter-spacing: 1.3px;
    margin-bottom: 13px;
}

#result-message textarea {
    background: #f5faf8 !important;
    border: 1px solid #e0eeea !important;
    border-radius: 15px !important;
    color: #254641 !important;
    font-size: 15px !important;
    line-height: 1.8 !important;
    padding: 18px !important;
}

#info-strip {
    background: #e8f5f1;
    border: 1px solid #d6ebe4;
    border-radius: 16px;
    padding: 18px 20px;
    color: #315d54;
    font-size: 13px;
    line-height: 1.7;
    margin-top: 20px;
}

#footer {
    text-align: center;
    color: #7c8d88;
    font-size: 12px;
    line-height: 1.8;
    padding: 24px 10px 8px;
}

@media (max-width: 700px) {
    #hero {
        padding: 25px 22px;
        border-radius: 20px;
    }
    #brand {
        font-size: 29px;
    }
    #main-card, #result-card {
        padding: 17px;
        border-radius: 18px;
    }
}
"""


# ---------------- WEBSITE LAYOUT ----------------

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="emerald",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("DM Sans"), "Arial", "sans-serif"]
    ),
    css=custom_css,
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.HTML("""
    <div id="hero">
        <div id="hero-tag">AI-ASSISTED HEALTH TECHNOLOGY</div>
        <h1 id="brand">✳ SkinSakhi</h1>
        <div id="hero-subtitle">
            A simple, accessible screening prototype designed to explore
            AI-assisted skin condition classification. Upload an image
            to receive an experimental model suggestion.
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=1, elem_id="main-card"):
            gr.HTML("""
            <div id="section-title">Image screening</div>
            <div id="section-help">
                Upload a clear, well-lit skin photograph.
                For best results, avoid filters, screenshots and blurry images.
            </div>
            """)

            image_input = gr.Image(
                type="pil",
                label="Upload a photo",
                elem_id="upload-image",
                height=320
            )

            analyze_btn = gr.Button(
                "Analyze image  →",
                variant="primary",
                elem_id="analyze-btn"
            )

            clear_btn = gr.ClearButton(
                [image_input],
                value="Clear image",
                elem_id="clear-btn"
            )

        with gr.Column(scale=1, elem_id="result-card"):
            gr.HTML("""
            <div id="section-title">Your screening result</div>
            <div id="section-help">
                Results appear here after image processing.
            </div>
            """)

            result_title = gr.Textbox(
                value="READY TO BEGIN",
                label="STATUS",
                interactive=False,
                elem_id="result-label"
            )

            result_message = gr.Textbox(
                value="Upload an image and select Analyze image to begin.",
                label="",
                lines=7,
                interactive=False,
                elem_id="result-message"
            )

            gr.HTML("""
            <div id="info-strip">
                <b>Important</b><br>
                This experimental tool is not a diagnostic device.
                Its suggestions can be incorrect. A qualified healthcare
                professional should assess any skin concern.
            </div>
            """)

    analyze_btn.click(
        fn=update_result,
        inputs=image_input,
        outputs=[result_title, result_message, result_message]
    )

    gr.HTML("""
    <div id="footer">
        <b>SkinSakhi</b> · AI-assisted skin screening prototype<br>
        Built for learning and early-stage research. Not for medical diagnosis.
    </div>
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/1818230317.py:324: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a68eceb44b57917e5f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [25]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.70
MIN_MARGIN = 0.15

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Model classes and labels do not match.")


def basic_image_check(image):
    arr = np.asarray(
        image.convert("RGB").resize((160, 160)),
        dtype=np.float32
    )
    variation = float(np.mean(np.std(arr, axis=(0, 1))))
    white = float(np.mean(np.all(arr > 235, axis=2)))
    dark = float(np.mean(np.all(arr < 35, axis=2)))

    if variation < 12 or dark > 0.85:
        return False
    if white > 0.82 and variation < 35:
        return False
    return True


def screen_skin(image):
    if image is None:
        return "Please upload a skin image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer photo."

    if not basic_image_check(image):
        return (
            "IMAGE NOT ACCEPTED\n\n"
            "This may be a document or unsuitable image. "
            "Please upload a clear skin photo."
        )

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]
    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try again."

    indices = np.argsort(predictions)[::-1]
    best = int(indices[0])
    confidence = float(predictions[best])
    second = float(predictions[indices[1]]) if len(indices) > 1 else 0.0

    if confidence < THRESHOLD or confidence - second < MIN_MARGIN:
        return (
            "RESULT UNCERTAIN\n\n"
            "The model could not provide a sufficiently clear result. "
            "Try a clearer photo or consult a qualified healthcare professional."
        )

    return (
        "EXPERIMENTAL SCREENING SUGGESTION\n\n"
        f"Suggested class: {class_names[best]}\n\n"
        "This is not a diagnosis. The model may be wrong, even when "
        "its score is high. Please consult a qualified healthcare professional."
    )


css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@500;600;700;800&display=swap');

.gradio-container {
    max-width: 1120px !important;
    margin: auto !important;
    background: #f3f8f6 !important;
    font-family: 'DM Sans', sans-serif !important;
}

body {
    background: #f3f8f6 !important;
}

#hero {
    background: linear-gradient(125deg, #073c38 0%, #087f75 70%, #16a394 100%);
    padding: 35px;
    border-radius: 24px;
    color: white;
    margin: 12px 0 22px;
    box-shadow: 0 12px 32px rgba(7, 80, 70, .15);
}

#hero h1 {
    color: white;
    font-family: 'Manrope', sans-serif;
    font-size: 36px;
    font-weight: 800;
    margin: 12px 0;
    letter-spacing: -1px;
}

#tag {
    display: inline-block;
    color: #e1fff8;
    background: rgba(255,255,255,.13);
    border: 1px solid rgba(255,255,255,.24);
    padding: 7px 12px;
    border-radius: 30px;
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1px;
}

#subtitle {
    color: #e0f5ef;
    max-width: 680px;
    line-height: 1.8;
    font-size: 15px;
}

.panel {
    background: white;
    border: 1px solid #e2ece8;
    border-radius: 20px;
    padding: 23px;
    box-shadow: 0 7px 25px rgba(22, 61, 51, .045);
}

.panel h2 {
    color: #153f39;
    font-family: 'Manrope', sans-serif;
    font-size: 21px;
    font-weight: 800;
    margin: 0 0 7px;
}

.hint {
    color: #71847e;
    font-size: 13px;
    line-height: 1.7;
    margin-bottom: 16px;
}

#upload {
    border: 2px dashed #b5d9d0 !important;
    border-radius: 15px !important;
    background: #f6fbf9 !important;
}

#analyze {
    background: linear-gradient(100deg, #087f75, #0da18f) !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    min-height: 51px !important;
    font-weight: 700 !important;
    box-shadow: 0 5px 14px rgba(8, 127, 117, .18);
    transition: .2s ease;
}

#analyze:hover {
    transform: translateY(-1px);
    box-shadow: 0 8px 18px rgba(8, 127, 117, .25);
}

#clear {
    border-radius: 12px !important;
    border: 1px solid #dce9e4 !important;
    background: #ffffff !important;
}

#result {
    background: #f6faf8 !important;
    border: 1px solid #e0ece7 !important;
    border-radius: 14px !important;
}

#result textarea {
    color: #284a42 !important;
    font-size: 15px !important;
    line-height: 1.85 !important;
    padding: 16px !important;
}

#notice {
    background: #eaf5f1;
    color: #315c52;
    border: 1px solid #d9eae3;
    padding: 15px 17px;
    border-radius: 13px;
    font-size: 12px;
    line-height: 1.8;
    margin-top: 15px;
}

#footer {
    color: #7a8c85;
    font-size: 12px;
    line-height: 1.8;
    text-align: center;
    padding: 22px 8px;
}

@media (max-width: 700px) {
    #hero { padding: 24px; border-radius: 18px; }
    #hero h1 { font-size: 30px; }
    .panel { padding: 16px; }
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="emerald",
        neutral_hue="slate"
    ),
    css=css,
    title="SkinSakhi | Skin Screening"
) as demo:

    gr.HTML("""
    <div id="hero">
        <div id="tag">AI-ASSISTED SKIN SCREENING</div>
        <h1>✳ SkinSakhi</h1>
        <div id="subtitle">
            A simple, accessible screening prototype exploring AI-assisted
            skin condition classification. Upload a clear photograph to
            receive an experimental model suggestion.
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            with gr.Group(elem_classes=["panel"]):
                gr.HTML("""
                <h2>Image screening</h2>
                <div class="hint">
                    Upload a clear, well-lit skin photograph.
                    Avoid blurry images, filters and screenshots.
                </div>
                """)

                image_input = gr.Image(
                    type="pil",
                    label="Upload skin photo",
                    elem_id="upload",
                    height=310
                )

                analyze = gr.Button(
                    "Analyze image  →",
                    variant="primary",
                    elem_id="analyze"
                )

                gr.ClearButton(
                    [image_input],
                    value="Clear image",
                    elem_id="clear"
                )

        with gr.Column(scale=1):
            with gr.Group(elem_classes=["panel"]):
                gr.HTML("""
                <h2>Your screening result</h2>
                <div class="hint">
                    Your result will appear here after analysis.
                </div>
                """)

                result = gr.Textbox(
                    value="Ready to begin\n\nUpload an image and select Analyze image.",
                    label="RESULT",
                    lines=8,
                    interactive=False,
                    elem_id="result"
                )

                gr.HTML("""
                <div id="notice">
                    <b>Medical disclaimer</b><br>
                    SkinSakhi is an experimental educational prototype,
                    not a medical device. It cannot diagnose or rule out
                    disease. Predictions can be wrong. Consult a qualified
                    healthcare professional for medical advice.
                </div>
                """)

    analyze.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result
    )

    gr.HTML("""
    <div id="footer">
        <b>SkinSakhi</b> · AI-assisted skin screening prototype<br>
        Built for learning and early-stage research · Not for diagnosis
    </div>
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/2034472784.py:223: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://05ef91637e09efa8ac.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.70
MIN_MARGIN = 0.15

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


def basic_image_check(image):
    small = np.asarray(
        image.convert("RGB").resize((160, 160)),
        dtype=np.float32
    )

    colour_variation = float(np.mean(np.std(small, axis=(0, 1))))
    near_white = float(np.mean(np.all(small > 235, axis=2)))
    dark_pixels = float(np.mean(np.all(small < 35, axis=2)))

    if colour_variation < 12:
        return False

    if near_white > 0.82 and colour_variation < 35:
        return False

    if dark_pixels > 0.85:
        return False

    return True


def screen_skin(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer image."

    if not basic_image_check(image):
        return (
            "IMAGE NOT ACCEPTED\n\n"
            "This image may be unsuitable or document-like. "
            "Please upload a clear skin photo.\n\n"
            "This basic check is not perfect and may make mistakes."
        )

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try another."

    if len(predictions) != len(class_names):
        return "Unable to assess this image."

    top_indices = np.argsort(predictions)[::-1]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )
    margin = confidence - second_score

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        return (
            "UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain about this image. "
            "Try a clearer skin photo or consult a healthcare professional."
        )

    return (
        "EXPERIMENTAL MODEL SUGGESTION\n\n"
        f"Suggested class: {class_names[top_index]}\n\n"
        "This is not a diagnosis. The model can still be wrong, "
        "even when its score is high. Please consult a qualified "
        "healthcare professional for medical advice."
    )


# Premium website styling
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@500;600;700;800&display=swap');

.gradio-container {
    max-width: 1120px !important;
    margin: auto !important;
    font-family: 'DM Sans', sans-serif !important;
    background: #f3f8f6 !important;
}

body {
    background: #f3f8f6 !important;
}

#hero {
    background: linear-gradient(125deg, #073c38 0%, #087f75 68%, #16a394 100%);
    border-radius: 25px;
    padding: 34px;
    margin: 12px 0 22px;
    color: white;
    box-shadow: 0 12px 32px rgba(7, 80, 70, .15);
}

#brand {
    color: white;
    font-family: 'Manrope', sans-serif;
    font-size: 36px;
    font-weight: 800;
    letter-spacing: -1px;
    margin: 13px 0 9px;
}

#tag {
    display: inline-block;
    color: #e3fff8;
    background: rgba(255,255,255,.13);
    border: 1px solid rgba(255,255,255,.25);
    border-radius: 30px;
    padding: 7px 12px;
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1px;
}

#hero-description {
    color: #e0f5ef;
    font-size: 15px;
    line-height: 1.8;
    max-width: 690px;
    margin-top: 10px;
}

.panel {
    background: white;
    border: 1px solid #e1ece7;
    border-radius: 21px;
    padding: 22px;
    box-shadow: 0 7px 25px rgba(22, 61, 51, .045);
}

.section-title {
    font-family: 'Manrope', sans-serif;
    color: #153f39;
    font-size: 21px;
    font-weight: 800;
    margin: 0 0 7px;
}

.section-description {
    color: #71847e;
    font-size: 13px;
    line-height: 1.7;
    margin-bottom: 17px;
}

#upload {
    border: 2px dashed #b5d9d0 !important;
    border-radius: 16px !important;
    background: #f6fbf9 !important;
    overflow: hidden;
}

#analyze {
    background: linear-gradient(100deg, #087f75, #0da18f) !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    min-height: 51px !important;
    font-size: 15px !important;
    font-weight: 700 !important;
    box-shadow: 0 5px 14px rgba(8, 127, 117, .18);
    transition: .2s ease;
}

#analyze:hover {
    transform: translateY(-1px);
    box-shadow: 0 8px 18px rgba(8, 127, 117, .25);
}

#clear {
    border-radius: 12px !important;
    border: 1px solid #dce9e4 !important;
    background: #ffffff !important;
}

#result {
    background: #f6faf8 !important;
    border: 1px solid #e0ece7 !important;
    border-radius: 14px !important;
}

#result textarea {
    color: #284a42 !important;
    font-size: 15px !important;
    line-height: 1.85 !important;
    padding: 16px !important;
}

#medical-note {
    background: #eaf5f1;
    color: #315c52;
    border: 1px solid #d9eae3;
    padding: 15px 17px;
    border-radius: 14px;
    font-size: 12px;
    line-height: 1.8;
    margin-top: 17px;
}

#footer {
    text-align: center;
    color: #7a8c85;
    font-size: 12px;
    line-height: 1.8;
    padding: 23px 8px 10px;
}

@media (max-width: 700px) {
    #hero {
        padding: 24px;
        border-radius: 19px;
    }

    #brand {
        font-size: 30px;
    }

    .panel {
        padding: 16px;
        border-radius: 18px;
    }
}
"""


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    css=custom_css,
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.HTML("""
    <div id="hero">
        <div id="tag">AI-ASSISTED SKIN SCREENING</div>
        <h1 id="brand">✳ SkinSakhi</h1>
        <div id="hero-description">
            A simple, accessible screening prototype exploring AI-assisted
            skin condition classification. Upload a clear skin photograph
            to receive an experimental model suggestion.
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):

        with gr.Column(scale=1):
            with gr.Group(elem_classes=["panel"]):
                gr.HTML("""
                <div class="section-title">Image screening</div>
                <div class="section-description">
                    Upload a clear, well-lit skin photograph.
                    Avoid blurry images, filters and screenshots.
                </div>
                """)

                image_input = gr.Image(
                    type="pil",
                    label="Upload skin photo",
                    elem_id="upload",
                    height=310
                )

                analyze_btn = gr.Button(
                    "Analyze Image  →",
                    variant="primary",
                    elem_id="analyze"
                )

                gr.ClearButton(
                    [image_input],
                    value="Clear Image",
                    elem_id="clear"
                )

        with gr.Column(scale=1):
            with gr.Group(elem_classes=["panel"]):
                gr.HTML("""
                <div class="section-title">Your screening result</div>
                <div class="section-description">
                    Your result will appear here after analysis.
                </div>
                """)

                result_output = gr.Textbox(
                    value="Ready to begin\n\nUpload an image and select Analyze Image.",
                    label="SCREENING RESULT",
                    lines=9,
                    interactive=False,
                    elem_id="result"
                )

                gr.HTML("""
                <div id="medical-note">
                    <b>Medical disclaimer</b><br>
                    SkinSakhi is an experimental educational prototype,
                    not a medical device. It cannot diagnose or rule out
                    disease. Predictions may be wrong. Consult a qualified
                    healthcare professional for medical advice.
                </div>
                """)

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result_output
    )

    gr.HTML("""
    <div id="footer">
        <b>SkinSakhi</b> · AI-assisted skin screening prototype<br>
        Built for learning and early-stage research · Not for diagnosis
    </div>
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/1456880929.py:261: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9f61e1c12f6aac399a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.70
MIN_MARGIN = 0.15

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


def basic_image_check(image):
    small = np.asarray(
        image.convert("RGB").resize((160, 160)),
        dtype=np.float32
    )

    colour_variation = float(np.mean(np.std(small, axis=(0, 1))))
    near_white = float(np.mean(np.all(small > 235, axis=2)))
    dark_pixels = float(np.mean(np.all(small < 35, axis=2)))

    if colour_variation < 12:
        return False

    if near_white > 0.82 and colour_variation < 35:
        return False

    if dark_pixels > 0.85:
        return False

    return True


def screen_skin(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer image."

    if not basic_image_check(image):
        return (
            "IMAGE NOT ACCEPTED\n\n"
            "This image may be unsuitable or document-like. "
            "Please upload a clear skin photo.\n\n"
            "This basic check is not perfect and may make mistakes."
        )

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try another."

    if len(predictions) != len(class_names):
        return "Unable to assess this image."

    top_indices = np.argsort(predictions)[::-1]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )
    margin = confidence - second_score

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        return (
            "UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain about this image. "
            "Try a clearer skin photo or consult a healthcare professional."
        )

    return (
        "EXPERIMENTAL MODEL SUGGESTION\n\n"
        f"Suggested class: {class_names[top_index]}\n\n"
        "This is not a diagnosis. The model can still be wrong, "
        "even when its score is high. Please consult a qualified "
        "healthcare professional for medical advice."
    )


custom_css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@500;600;700;800&display=swap');

.gradio-container {
    max-width: 1120px !important;
    margin: auto !important;
    font-family: 'DM Sans', sans-serif !important;
    background: #f3f8f6 !important;
}

body {
    background: #f3f8f6 !important;
}

#hero {
    background: linear-gradient(125deg, #073c38 0%, #087f75 68%, #16a394 100%);
    border-radius: 25px;
    padding: 34px;
    margin: 12px 0 22px;
    color: white;
    box-shadow: 0 12px 32px rgba(7, 80, 70, .15);
}

#brand {
    color: white;
    font-family: 'Manrope', sans-serif;
    font-size: 36px;
    font-weight: 800;
    letter-spacing: -1px;
    margin: 13px 0 9px;
}

#tag {
    display: inline-block;
    color: #e3fff8;
    background: rgba(255,255,255,.13);
    border: 1px solid rgba(255,255,255,.25);
    border-radius: 30px;
    padding: 7px 12px;
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1px;
}

#hero-description {
    color: #e0f5ef;
    font-size: 15px;
    line-height: 1.8;
    max-width: 690px;
    margin-top: 10px;
}

.panel {
    background: white;
    border: 1px solid #e1ece7;
    border-radius: 21px;
    padding: 22px;
    box-shadow: 0 7px 25px rgba(22, 61, 51, .045);
}

.section-title {
    font-family: 'Manrope', sans-serif;
    color: #153f39;
    font-size: 21px;
    font-weight: 800;
    margin: 0 0 7px;
}

.section-description {
    color: #71847e;
    font-size: 13px;
    line-height: 1.7;
    margin-bottom: 17px;
}

#upload {
    border: 2px dashed #b5d9d0 !important;
    border-radius: 16px !important;
    background: #f6fbf9 !important;
    overflow: hidden;
}

#analyze {
    background: linear-gradient(100deg, #087f75, #0da18f) !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    min-height: 51px !important;
    font-size: 15px !important;
    font-weight: 700 !important;
    box-shadow: 0 5px 14px rgba(8, 127, 117, .18);
    transition: .2s ease;
}

#analyze:hover {
    transform: translateY(-1px);
    box-shadow: 0 8px 18px rgba(8, 127, 117, .25);
}

#clear {
    border-radius: 12px !important;
    border: 1px solid #dce9e4 !important;
    background: #ffffff !important;
}

#result {
    background: #f6faf8 !important;
    border: 1px solid #e0ece7 !important;
    border-radius: 14px !important;
}

#result textarea {
    color: #284a42 !important;
    font-size: 15px !important;
    line-height: 1.85 !important;
    padding: 16px !important;
}

#medical-note {
    background: #eaf5f1;
    color: #315c52;
    border: 1px solid #d9eae3;
    padding: 15px 17px;
    border-radius: 14px;
    font-size: 12px;
    line-height: 1.8;
    margin-top: 17px;
}

#footer {
    text-align: center;
    color: #7a8c85;
    font-size: 12px;
    line-height: 1.8;
    padding: 23px 8px 10px;
}

@media (max-width: 700px) {
    #hero {
        padding: 24px;
        border-radius: 19px;
    }

    #brand {
        font-size: 30px;
    }

    .panel {
        padding: 16px;
        border-radius: 18px;
    }
}
"""


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    css=custom_css,
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.HTML("""
    <div id="hero">
        <div id="tag">AI-ASSISTED SKIN SCREENING</div>
        <h1 id="brand">✳ SkinSakhi</h1>
        <div id="hero-description">
            A simple, accessible screening prototype exploring AI-assisted
            skin condition classification. Upload a clear skin photograph
            to receive an experimental model suggestion.
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):

        with gr.Column(scale=1):
            with gr.Group(elem_classes=["panel"]):
                gr.HTML("""
                <div class="section-title">Image screening</div>
                <div class="section-description">
                    Upload a clear, well-lit skin photograph.
                    Avoid blurry images, filters and screenshots.
                </div>
                """)

                image_input = gr.Image(
                    type="pil",
                    label="Upload skin photo",
                    elem_id="upload",
                    height=310
                )

                analyze_btn = gr.Button(
                    "Analyze Image  →",
                    variant="primary",
                    elem_id="analyze"
                )

                gr.ClearButton(
                    [image_input],
                    value="Clear Image",
                    elem_id="clear"
                )

        with gr.Column(scale=1):
            with gr.Group(elem_classes=["panel"]):
                gr.HTML("""
                <div class="section-title">Your screening result</div>
                <div class="section-description">
                    Your result will appear here after analysis.
                </div>
                """)

                result_output = gr.Textbox(
                    value="Ready to begin\n\nUpload an image and select Analyze Image.",
                    label="SCREENING RESULT",
                    lines=9,
                    interactive=False,
                    elem_id="result"
                )

                gr.HTML("""
                <div id="medical-note">
                    <b>Medical disclaimer</b><br>
                    SkinSakhi is an experimental educational prototype,
                    not a medical device. It cannot diagnose or rule out
                    disease. Predictions may be wrong. Consult a qualified
                    healthcare professional for medical advice.
                </div>
                """)

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result_output
    )

    gr.HTML("""
    <div id="footer">
        <b>SkinSakhi</b> · AI-assisted skin screening prototype<br>
        Built for learning and early-stage research · Not for diagnosis
    </div>
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/787507169.py:260: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3b99f9d160debae1a6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [28]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.70
MIN_MARGIN = 0.15

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


def basic_image_check(image):
    small = np.asarray(
        image.convert("RGB").resize((160, 160)),
        dtype=np.float32
    )

    colour_variation = float(np.mean(np.std(small, axis=(0, 1))))
    near_white = float(np.mean(np.all(small > 235, axis=2)))
    dark_pixels = float(np.mean(np.all(small < 35, axis=2)))

    # Heuristic only; this is not a trained skin-image detector.
    if colour_variation < 12:
        return False

    if near_white > 0.82 and colour_variation < 35:
        return False

    if dark_pixels > 0.85:
        return False

    return True


def screen_skin(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer image."

    if not basic_image_check(image):
        return (
            "IMAGE NOT ACCEPTED\n\n"
            "This image may be unsuitable or document-like. "
            "Please upload a clear skin photo.\n\n"
            "This basic check is not perfect and may make mistakes."
        )

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try another."

    if len(predictions) != len(class_names):
        return "Unable to assess this image."

    top_indices = np.argsort(predictions)[::-1]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )
    margin = confidence - second_score

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        return (
            "UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain about this image. "
            "Try a clearer skin photo or consult a healthcare professional."
        )

    return (
        "EXPERIMENTAL MODEL SUGGESTION\n\n"
        f"Suggested class: {class_names[top_index]}\n\n"
        "This is not a diagnosis. The model can still be wrong, "
        "even when its score is high. Please consult a qualified "
        "healthcare professional for medical advice."
    )


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown("""
    # 🩺 SkinSakhi
    ### AI-Assisted Skin Condition Screening

    Upload a clear skin photo to receive an experimental model suggestion.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Skin Image"
            )

            analyze_btn = gr.Button(
                "Analyze Image",
                variant="primary"
            )

            clear_btn = gr.ClearButton([image_input])

        with gr.Column():
            result_output = gr.Textbox(
                label="Screening Result",
                lines=9,
                interactive=False
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result_output
    )

    gr.Markdown("""
    ---
    **Medical disclaimer:** SkinSakhi is an experimental educational
    prototype, not a medical device. It cannot diagnose or rule out
    disease. Unrelated images may still pass the basic image check.
    Seek professional medical advice.
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/4009253520.py:106: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc7ebbe9b508cd2901.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

THRESHOLD = 0.50
MIN_MARGIN = 0.10

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


def basic_image_check(image):
    small = np.asarray(
        image.convert("RGB").resize((160, 160)),
        dtype=np.float32
    )

    colour_variation = float(np.mean(np.std(small, axis=(0, 1))))
    near_white = float(np.mean(np.all(small > 235, axis=2)))
    dark_pixels = float(np.mean(np.all(small < 35, axis=2)))

    if colour_variation < 12:
        return False

    if near_white > 0.82 and colour_variation < 35:
        return False

    if dark_pixels > 0.85:
        return False

    return True


def screen_skin(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer image."

    if not basic_image_check(image):
        return (
            "IMAGE NOT ACCEPTED\n\n"
            "This image may be unsuitable or document-like. "
            "Please upload a clear skin photo.\n\n"
            "This basic check is not perfect and may make mistakes."
        )

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try another."

    if len(predictions) != len(class_names):
        return "Unable to assess this image."

    top_indices = np.argsort(predictions)[::-1]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )
    margin = confidence - second_score

    if confidence < THRESHOLD or margin < MIN_MARGIN:
        return (
            "UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain about this image. "
            "Try a clearer skin photo or consult a healthcare professional."
        )

    return (
        "EXPERIMENTAL MODEL SUGGESTION\n\n"
        f"Suggested class: {class_names[top_index]}\n\n"
        "This is not a diagnosis. The model can still be wrong, "
        "even when its score is high. Please consult a qualified "
        "healthcare professional for medical advice."
    )


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown("""
    # 🩺 SkinSakhi
    ### AI-Assisted Skin Condition Screening

    Upload a clear skin photo to receive an experimental model suggestion.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Skin Image"
            )

            analyze_btn = gr.Button(
                "Analyze Image",
                variant="primary"
            )

            clear_btn = gr.ClearButton([image_input])

        with gr.Column():
            result_output = gr.Textbox(
                label="Screening Result",
                lines=9,
                interactive=False
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result_output
    )

    gr.Markdown("""
    ---
    **Medical disclaimer:** SkinSakhi is an experimental educational
    prototype, not a medical device. It cannot diagnose or rule out
    disease. Unrelated images may still pass the basic image check.
    Seek professional medical advice.
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/2528933392.py:105: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://164ac96465d5e778a1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [30]:

import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


def screen_skin(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer image."

    image = image.resize((224, 224))
    arr = np.asarray(image, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Invalid model output. Please check the model."

    if len(predictions) != len(class_names):
        return "Model output does not match the class labels."

    top_indices = np.argsort(predictions)[::-1][:3]
    top_index = int(top_indices[0])

    print("\nDEBUG - Top 3 model predictions:")
    for i in top_indices:
        print(f"{class_names[i]}: {predictions[i]:.4f}")

    return (
        "EXPERIMENTAL MODEL SUGGESTION\n\n"
        f"Suggested class: {class_names[top_index]}\n\n"
        "Top model predictions:\n"
        + "\n".join(
            f"{class_names[i]}: {predictions[i]:.4f}"
            for i in top_indices
        )
        + "\n\nThis is not a diagnosis. The model can be wrong. "
        "Please consult a qualified healthcare professional."
    )


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown("""
    # 🩺 SkinSakhi
    ### AI-Assisted Skin Condition Screening

    Upload a clear skin photo to receive an experimental model suggestion.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Skin Image"
            )

            analyze_btn = gr.Button(
                "Analyze Image",
                variant="primary"
            )

            clear_btn = gr.ClearButton([image_input])

        with gr.Column():
            result_output = gr.Textbox(
                label="Screening Result",
                lines=10,
                interactive=False
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result_output
    )

    gr.Markdown("""
    ---
    **Medical disclaimer:** SkinSakhi is an experimental educational
    prototype, not a medical device. It cannot diagnose or rule out
    disease. Predictions are not clinically validated. Seek professional
    medical advice.
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/1929678609.py:66: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5e19a69dc901f0f62.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [31]:

import os
import cv2
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


def is_document_or_paper(image):
    image = image.convert("RGB").resize((224, 224))
    rgb = np.asarray(image, dtype=np.uint8)

    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    brightness = gray > 215
    low_saturation = hsv[:, :, 1] < 55

    bright_paper = brightness & low_saturation
    paper_ratio = float(np.mean(bright_paper))

    edges = cv2.Canny(gray, 60, 150)
    edge_ratio = float(np.mean(edges > 0))

    # Reject images dominated by a bright, low-colour paper background
    # with visible text or line details.
    if paper_ratio > 0.45 and edge_ratio > 0.025:
        return True

    # Reject nearly blank white or grey document images.
    if paper_ratio > 0.80 and float(np.std(gray)) < 35:
        return True

    return False


def screen_skin(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer image."

    if is_document_or_paper(image):
        return (
            "IMAGE NOT RELATED TO SKIN\n\n"
            "This image appears to contain paper, notes, or a document.\n\n"
            "Please upload a clear photo of a skin condition."
        )

    resized = image.resize((224, 224))
    arr = np.asarray(resized, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try another."

    top_indices = np.argsort(predictions)[::-1]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )

    if confidence < 0.50 or confidence - second_score < 0.10:
        return (
            "UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain about this image.\n"
            "Please upload a clearer skin photo or consult "
            "a healthcare professional."
        )

    return (
        "EXPERIMENTAL MODEL SUGGESTION\n\n"
        f"Suggested class: {class_names[top_index]}\n\n"
        "This is not a diagnosis. The model may be wrong, "
        "even when its score is high. Please consult a qualified "
        "healthcare professional for medical advice."
    )


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown("""
    # 🩺 SkinSakhi
    ### AI-Assisted Skin Condition Screening

    Upload a clear skin photo to receive an experimental model suggestion.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Skin Image"
            )

            analyze_btn = gr.Button(
                "Analyze Image",
                variant="primary"
            )

            clear_btn = gr.ClearButton([image_input])

        with gr.Column():
            result_output = gr.Textbox(
                label="Screening Result",
                lines=9,
                interactive=False
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result_output
    )

    gr.Markdown("""
    ---
    **Medical disclaimer:** SkinSakhi is an experimental educational
    prototype, not a medical device. It cannot diagnose or rule out
    disease. The image-type check is heuristic and may make mistakes.
    Seek professional medical advice.
    """)

demo.launch(share=True)


/tmp/ipykernel_1404/18691712.py:105: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cef3bf7798067968f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
